In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import(
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    recall_score,
    precision_score,
    f1_score,
    roc_auc_score
)

#Models to compare
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from isotree import IsolationForest as EIForest

#I will try to import the optional libraries 

try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
    print("XGBoost is available")
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost is NOT available. You need to install it: pip install xgboost")


try:
    import lightgbm as lgb
    LIGHTGBM_AVAILABLE = True
    print("LIGHTGBM is available")
except ImportError:
    XGBOOST_AVAILABLE = False
    print("LIGHTGBM is NOT available. You need to install it: pip install lightgbm")


sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (15,7)



XGBoost is available
LIGHTGBM is available


In [2]:
def prepare_data(data_path):
    """
    Esta función carga el dataset, crea todas las características 
    (temporales y de ventana), realiza la división cronológica y escala los datos.
    Devuelve todos los conjuntos de datos necesarios para entrenar y evaluar.
    """
    print("\n--- Iniciando preparación de datos ---")
    df = pd.read_csv(data_path)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    # ---- Feature Engineering ----
    df['hour'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    
    df = df.sort_values(['household_id', 'timestamp']).reset_index(drop=True)
    
    window_sizes = [4, 8, 24]
    grouped = df.groupby('household_id')['consumption_l']
    for window in window_sizes:
        rolling_mean = grouped.transform(lambda x: x.rolling(window=window, min_periods=1).mean())
        rolling_std = grouped.transform(lambda x: x.rolling(window=window, min_periods=1).std())
        df[f'rolling_cv_{window}'] = rolling_std / (rolling_mean + 1e-6)
        df[f'rolling_mean_{window}'] = rolling_mean
        df[f'rolling_std_{window}'] = rolling_std
    df.fillna(0, inplace=True)
    
    # ---- Selección de Características (CORREGIDA) ----
    feature_cols = [
        'consumption_l', 'hour', 'day_of_week', 'is_weekend',
        'rolling_mean_4', 'rolling_std_4', 'rolling_cv_4',  # <-- Corregido de 'csv' a 'cv'
        'rolling_mean_8', 'rolling_std_8', 'rolling_cv_8',  # <-- Corregido de 'csv' a 'cv'
        'rolling_mean_24', 'rolling_std_24', 'rolling_cv_24' # <-- Corregido de 'csv' a 'cv'
    ]
    target = 'is_leak'
    
    # ---- División Cronológica ----
    df = df.sort_values('timestamp')
    split_point = int(len(df) * 0.8)
    train_df = df.iloc[:split_point]
    test_df = df.iloc[split_point:]
    
    X_train = train_df[feature_cols]
    y_train = train_df[target]
    X_test = test_df[feature_cols]
    y_test = test_df[target]
    
    # ---- Escalado de Datos ----
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    print(f"✓ Datos preparados.")
    
    return X_train, X_train_scaled, X_test, X_test_scaled, y_train, y_test, scaler, feature_cols

In [3]:
def evaluate_model(model, X_test, y_test, model_name, is_unsupervised=False):
    """
    Evalúa cualquier modelo y devuelve un diccionario de resultados.
    """
    if is_unsupervised:
        y_pred = model.predict(X_test)
        y_pred_mapped = np.where(y_pred == -1, 1, 0)
    else: # Modelos Supervisados
        y_pred_mapped = model.predict(X_test)

    # Crear el diccionario de resultados base
    results = {
        'model_name': model_name,
        'recall': recall_score(y_test, y_pred_mapped),
        'precision': precision_score(y_test, y_pred_mapped, zero_division=0),
        'f1': f1_score(y_test, y_pred_mapped, zero_division=0),
        'auc': None # Empezar con AUC como None
    }
    
    # Intentar calcular AUC-ROC solo si es posible
    if not is_unsupervised and hasattr(model, 'predict_proba'):
        try:
            y_pred_proba = model.predict_proba(X_test)[:, 1]
            results['auc'] = roc_auc_score(y_test, y_pred_proba)
        except Exception:
            pass # Si falla, 'auc' simplemente se quedará como None
    
    return results

def train_isolation_forest(X_train_scaled, X_test_scaled, y_test, contamination=0.03):
    """Entrena y evalúa un modelo Isolation Forest de scikit-learn."""
    print(f"\n--- 1. Entrenando Isolation Forest (No Supervisado) ---")
    model = IsolationForest(
        n_estimators=200, 
        contamination=contamination, 
        random_state=42, 
        n_jobs=-1
    )
    model.fit(X_train_scaled)
    results = evaluate_model(model, X_test_scaled, y_test, "Isolation Forest", is_unsupervised=True)
    return model, results

def train_local_outlier_factor(X_train_scaled, X_test_scaled, y_test, contamination=0.02):
    print("\n--- Training Local Outlier Factor (Non Supervised) ---")
    model = LocalOutlierFactor(n_neighbors=20, contamination=contamination, novelty=True, n_jobs=-1)
    model.fit(X_train_scaled)
    results = evaluate_model(model, X_test_scaled, y_test, "Local Outlier Factor", is_unsupervised=True)
    return model, results

def train_random_forest(X_train, X_test, y_train, y_test):
    print(f"\n--- Training Random Forest (Supervised) ---")

    model = RandomForestClassifier(
        n_estimators = 100,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    results=evaluate_model(model, X_test, y_test, "Random Forest", is_unsupervised=False)
    return model, results

def train_isotree_eif(X_train_scaled, X_test_scaled, y_test, contamination=0.03):
    """
    Entrena y evalúa un Extended Isolation Forest usando la librería 'isotree'.
    Esta versión es compatible con las últimas actualizaciones de la librería.
    """
    print(f"\n--- 7. Entrenando Extended Isolation Forest (isotree) ---")
    
    # La inicialización del modelo ahora no necesita 'new_models' o 'prob_outliers'.
    # El parámetro 'contamination' se pasa directamente aquí.
    model = EIForest(
        ntrees=200,
        nthreads=-1,
        contamination=0.02 # Pasamos la contaminación directamente
    )
    
    # Entrenar el modelo
    model.fit(X_train_scaled)
    
    # --- PREDICCIÓN CON LA SINTAXIS CORRECTA ---
    # El método .predict() de la versión actual de 'isotree' devuelve -1 para anomalía y 1 para normal
    # cuando se le ha pasado un valor de 'contamination' al inicializar.
    # Esto es EXACTAMENTE lo que nuestra función 'evaluate_model' espera.
    
    # Evaluar y devolver los resultados
    results = evaluate_model(
        model, 
        X_test_scaled, 
        y_test, 
        "EIF (isotree)", 
        is_unsupervised=True
    )
    
    return model, results

def train_xgboost(X_train, X_test, y_train, y_test):
    """Entrena XGBoost con manejo de desbalance"""
    if not XGBOOST_AVAILABLE:
        return None, None
    
    print(f"\n{'='*80}")
    print("3. XGBoost (Supervisado)")
    print(f"{'='*80}")
    
    # Calcular scale_pos_weight para manejar desbalance
    n_samples = len(y_train)
    n_leaks = y_train.sum()
    n_normal = n_samples - n_leaks
    scale_pos_weight = n_normal / n_leaks
    
    model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=10,
        learning_rate=0.1,
        scale_pos_weight=scale_pos_weight,  # Manejo de desbalance
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss',
        verbosity=0
    )
    
    print("Entrenando XGBoost...")
    model.fit(X_train, y_train)
    results = evaluate_model(model, X_test, y_test, "XGBoost", is_unsupervised=False)
    
    return model, results

def train_lightgbm(X_train, X_test, y_train, y_test):
    """Entrena LightGBM con manejo de desbalance"""
    if not LIGHTGBM_AVAILABLE:
        return None, None
    
    print(f"\n{'='*80}")
    print("4. LightGBM (Supervisado)")
    print(f"{'='*80}")
    
    # Calcular scale_pos_weight
    n_samples = len(y_train)
    n_leaks = y_train.sum()
    n_normal = n_samples - n_leaks
    scale_pos_weight = n_normal / n_leaks
    
    model = lgb.LGBMClassifier(
        n_estimators=200,
        max_depth=10,
        learning_rate=0.1,
        scale_pos_weight=scale_pos_weight,  # Manejo de desbalance
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    
    print("Entrenando LightGBM...")
    model.fit(X_train, y_train)
    results = evaluate_model(model, X_test, y_test, "LightGBM", is_unsupervised=False)
    
    return model, results


def train_oneclass_svm(X_train_scaled, X_test_scaled, y_test, nu=0.02):
    """Entrena One-Class SVM"""
    print(f"\n{'='*80}")
    print("5. One-Class SVM (No Supervisado)")
    print(f"{'='*80}")
    
    model = OneClassSVM(
        nu=nu,  # Proporción esperada de outliers
        kernel='rbf',
        gamma='scale'
    )
    
    print("Entrenando One-Class SVM...")
    model.fit(X_train_scaled)
    results = evaluate_model(model, X_test_scaled, y_test, "One-Class SVM", is_unsupervised=True)
    
    return model, results

def train_lof(X_train_scaled, X_test_scaled, y_test, contamination=0.02):
    """Entrena Local Outlier Factor"""
    print(f"\n{'='*80}")
    print("6. Local Outlier Factor (No Supervisado)")
    print(f"{'='*80}")
    
    model = LocalOutlierFactor(
        n_neighbors=20,
        contamination=contamination,
        novelty=True,  # Permite predict en nuevos datos
        n_jobs=-1
    )
    
    print("Entrenando LOF...")
    model.fit(X_train_scaled)
    results = evaluate_model(model, X_test_scaled, y_test, "Local Outlier Factor", is_unsupervised=True)
    
    return model, results


def plot_comparison(results_list, output_path='model_comparison.png'):
    """Crea gráficos comparativos a partir de una lista de resultados."""
    results_df = pd.DataFrame(results_list)
    
    # Usaremos solo 2 subplots en lugar de 4 para simplificar.
    fig, axes = plt.subplots(1, 2, figsize=(20, 7))
    fig.suptitle('Comparación de Rendimiento de Modelos', fontsize=16)
    
    # --- Gráfico 1: Métricas principales (Recall, Precision, F1) ---
    x = np.arange(len(results_df))
    width = 0.25
    
    axes[0].bar(x - width, results_df['recall'], width, label='Recall', alpha=0.8)
    axes[0].bar(x, results_df['precision'], width, label='Precision', alpha=0.8)
    axes[0].bar(x + width, results_df['f1'], width, label='F1-Score', alpha=0.8)
    axes[0].set_xlabel('Modelo')
    axes[0].set_ylabel('Métrica')
    axes[0].set_title('Comparación de Métricas Principales')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(results_df['model_name'], rotation=45, ha='right')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='y')
    axes[0].set_ylim(0, 1.05) # Fijar el eje Y entre 0 y 1
    
    # --- Gráfico 2: Tabla de resultados ---
    axes[1].axis('tight')
    axes[1].axis('off')
    table_data = results_df[['model_name', 'recall', 'precision', 'f1', 'auc']].copy()
    # Rellenar valores NaN en AUC con un guion para que se vea bien en la tabla
    table_data['auc'] = table_data['auc'].fillna('-')
    table_data = table_data.round(4)

    table = axes[1].table(cellText=table_data.values,
                             colLabels=table_data.columns,
                             cellLoc='center',
                             loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.5)
    axes[1].set_title('Resumen Numérico de Resultados', pad=20)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96]) # Ajustar para el supertítulo
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"\n✓ Gráfico comparativo guardado en '{output_path}'")
    plt.close()    

In [4]:
# ==========================================================
# Celda Principal de Ejecución del Benchmark de Modelos
# ==========================================================

print("=" * 80)
print("INICIANDO COMPARACIÓN DE MODELOS PARA DETECCIÓN DE FUGAS")
print("=" * 80)

# --- 1. Preparar todos los datos de una sola vez ---
# (Asegúrate de que la función prepare_data está definida en una celda anterior)
DATA_PATH = '../../data/mixed_population_dataset_160_households.csv'
(X_train, X_train_scaled, X_test, X_test_scaled, 
 y_train, y_test, scaler, feature_cols) = prepare_data(DATA_PATH)

# Listas para guardar los resultados y los modelos entrenados
all_results = []
all_models = {}

# --- 2. Entrenar y evaluar cada modelo secuencialmente ---
# NOTA: Los valores de 'contamination' y 'nu' son hiperparámetros. 
# Se han elegido valores que funcionaron bien en experimentos previos.

# Modelo 1: Isolation Forest (No Supervisado)
model, results = train_isolation_forest(X_train_scaled, X_test_scaled, y_test, contamination=0.03)
if results: all_results.append(results); all_models[results['model_name']] = model

# Modelo 2: Random Forest (Supervisado)
model, results = train_random_forest(X_train, X_test, y_train, y_test)
if results: all_results.append(results); all_models[results['model_name']] = model

# Modelo 3: XGBoost (Supervisado)
model, results = train_xgboost(X_train, X_test, y_train, y_test)
if results: all_results.append(results); all_models[results['model_name']] = model

# Modelo 4: LightGBM (Supervisado)
model, results = train_lightgbm(X_train, X_test, y_train, y_test)
if results: all_results.append(results); all_models[results['model_name']] = model

# Modelo 5: One-Class SVM (No Supervisado) - PUEDE SER MUY LENTO
# print("\nADVERTENCIA: One-Class SVM puede tardar mucho tiempo en entrenar.")
# model, results = train_oneclass_svm(X_train_scaled, X_test_scaled, y_test, nu=0.03)
# if results: all_results.append(results); all_models[results['model_name']] = model

# Modelo 6: Local Outlier Factor (No Supervisado)
model, results = train_lof(X_train_scaled, X_test_scaled, y_test, contamination=0.03)
if results: all_results.append(results); all_models[results['model_name']] = model

#Modelo 7: EIF
model, results = train_isotree_eif(X_train_scaled, X_test_scaled, y_test, contamination=0.02)
if results:
    all_results.append(results)
    all_models[results['model_name']] = model


# --- 3. Mostrar y guardar el resumen de resultados ---
print("\n" + "=" * 80)
print("RESUMEN FINAL DE RESULTADOS DE LA COMPARACIÓN")
print("=" * 80)

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values('f1', ascending=False) # Ordenamos por F1-score

print("\nResultados ordenados por F1-Score:")
# Seleccionamos solo las columnas que existen en el DataFrame para evitar KeyErrors
cols_to_print = ['model_name', 'recall', 'precision', 'f1', 'auc']
existing_cols = [col for col in cols_to_print if col in results_df.columns]
print(results_df[existing_cols].to_string(index=False))

results_df.to_csv('model_comparison_results.csv', index=False)
print("\n✓ Resultados de la comparación guardados en 'model_comparison_results.csv'")

# --- 4. Generar gráficos comparativos ---
# (Asegúrate de que la función plot_comparison está definida)
if all_results:
    plot_comparison(all_results)

# --- 5. Reporte detallado del MEJOR modelo ---
if not results_df.empty:
    best_model_name = results_df.iloc[0]['model_name']
    # ... (el resto del código para el reporte final del mejor modelo iría aquí,
    # pero por ahora, la tabla comparativa es el resultado más importante)

print("\n" + "=" * 80)
print("ANÁLISIS COMPLETADO")
print("=" * 80)

INICIANDO COMPARACIÓN DE MODELOS PARA DETECCIÓN DE FUGAS

--- Iniciando preparación de datos ---
✓ Datos preparados.

--- 1. Entrenando Isolation Forest (No Supervisado) ---

--- Training Random Forest (Supervised) ---

3. XGBoost (Supervisado)
Entrenando XGBoost...

4. LightGBM (Supervisado)
Entrenando LightGBM...

6. Local Outlier Factor (No Supervisado)
Entrenando LOF...

--- 7. Entrenando Extended Isolation Forest (isotree) ---


TypeError: IsolationForest.__init__() got an unexpected keyword argument 'contamination'